In [85]:
from sympy import symbols, Rational
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
from codegen.codegen_utils import * 
init_printing()
printer = MyPrinter() 

# Z4c RHS evaluation
Throughout this notebook we assume:
$$
W = \Psi^{-2} \, ,
$$
So that
$$
\gamma_{ij} = W^{-2} \tilde{\gamma}_{ij}\, ,
$$
And
$$
\sqrt{\gamma} = W^{-3} \, .
$$
We first define some common terms and then the RHS. 

In [86]:
def der_scalar(s,name):
    _d = sp.Matrix([0]*3)
    for idir in range(3):
        _d[idir] = sp.symbols(f"d{name}_dx[{idir}]", real=True)
    return _d 

def upwind_der_scalar(s,name):
    return sp.symbols(f"d{name}_dx_upwind", real=True)

def second_der_scalar(s,name):
    out = [sp.MutableDenseMatrix([0]*3) for _ in range(3)] 
    icomp = 0 
    for idir in range(3):
        for jdir in range(idir,3):
            out[idir][jdir] = out[jdir][idir] = sp.symbols(f"dd{name}_dx2[{icomp}]", real=True)
            icomp += 1 
    return out

def der_vec(vec,name):
    out = [] 
    n = vec.shape[0]
    for idir in range(3):
        _out = sp.Matrix([0]*n)
        for i in range(n):
            _out[i] = sp.symbols(f"d{name}_dx[{ i + n * idir}]")
        out.append(_out)
    return out

def second_der_vec(vec, name):
    n = vec.shape[0]
    out = [[sp.Matrix([0]*n) for _ in range(3)] for _ in range(3)]

    icomp = 0
    for idir in range(3):
        for jdir in range(idir, 3):  # upper triangle only
            for i in range(n):
                out[idir][jdir][i] = out[jdir][idir][i] = sp.symbols(f"dd{name}_dx2[{icomp}]")
                icomp += 1
    return out

def upwind_der_vec(vec,name):
    
    n = vec.shape[0]
    out = sp.zeros(n,1)
    for i in range(n):
        out[i] = sp.symbols(f"d{name}_dx_upwind[{i}]")
    
    return out

def der_symm_tens(mat,name):
    rows, cols = mat.shape
    if rows != cols: raise ValueError("Non-square tensor cannot be symmetric")
    n = rows 
    ncomps = n * (n+1)//2 
    out = [] 
    for idir in range(3):
        _out = sp.MutableDenseMatrix(n, n, [0]* (n*n))
        icomp = 0 
        for i in range(n):
            for j in range(i,n):
                # order idir, i, j
                _out[i,j] = _out[j,i] = sp.symbols(f"d{name}_dx[{icomp + ncomps * idir}]")
                icomp += 1
        out.append(_out)
    return out

def upwind_der_symm_tens(mat,name):
    rows, cols = mat.shape
    if rows != cols: raise ValueError("Non-square tensor cannot be symmetric")
    n = rows 
    ncomps = n * (n+1)//2 
    out = sp.MutableDenseMatrix(n, n, [0]* (n*n))
    
    icomp = 0 
    for i in range(n):
        for j in range(i,n):
            # order idir, i, j
            out[i,j] = out[j,i] = sp.symbols(f"d{name}_dx_upwind[{icomp}]")
            icomp += 1
    return out

import sympy as sp

def second_der_symm_tens(mat, name):
    n = mat.shape[0]
    if n != mat.shape[1]:
        raise ValueError("Non-square tensor cannot be symmetric")
    ncomps = n * (n + 1) // 2  # unique components of tensor

    # Initialize 3x3 grid of n x n matrices
    out = [[sp.MutableDenseMatrix(n, n, [0]*(n*n)) for _ in range(3)] for _ in range(3)]

    icomp = 0
    for idir in range(3):
        for jdir in range(idir, 3):  # upper triangle in spatial indices
            _d = sp.MutableDenseMatrix(n, n, [0]*(n*n))
            for k in range(n):
                for l in range(k, n):  # upper triangle in tensor indices
                    symb = sp.symbols(f"dd{name}_dx2[{icomp}]")
                    _d[k, l] = _d[l, k] = symb  # tensor symmetry
                    icomp += 1
            out[idir][jdir] = _d
            out[jdir][idir] = _d  # enforce spatial symmetry
    return out  # out[idir][jdir][k,l] = ∂² A_{kl} / ∂x_idir ∂x_jdir

In [87]:
# variable block
gtxx,gtxy,gtxz,gtyy,gtyz,gtzz = symbols("gtdd[0] gtdd[1] gtdd[2] gtdd[3] gtdd[4] gtdd[5]") 
gtdd = sp.Matrix([
    [gtxx,gtxy,gtxz],
    [gtxy,gtyy,gtyz],
    [gtxz,gtyz,gtzz]
])


detg_expl = sp.det(gtdd)
detg = symbols("detg")
gtXX = (gtyy * gtzz - gtyz**2)/detg
gtYY = (gtxx * gtzz - gtxz**2)/detg 
gtZZ = (gtxx * gtyy - gtxy**2)/detg 
gtXY = (gtxz * gtyz - gtxy * gtzz)/detg 
gtXZ = (gtxy * gtyz - gtxz * gtyy)/detg 
gtYZ = (gtxy * gtxz - gtxx * gtyz)/detg

gtuu_expl = sp.Matrix([
    [gtXX,gtXY,gtXZ],
    [gtXY,gtYY,gtYZ],
    [gtXZ,gtYZ,gtZZ]
])

gtuu = sp.zeros(3,3)
cidx =0 
for a in range(3):
    for b in range(a,3):
        gtuu[a,b] = gtuu[b,a] = sp.symbols(f"gtuu[{cidx}]")
        cidx += 1 

print(sp.simplify(gtuu_expl.subs({detg: detg_expl}) * gtdd))

Atxx,Atxy,Atxz,Atyy,Atyz,Atzz = symbols("Atdd[0] Atdd[1] Atdd[2] Atdd[3] Atdd[4] Atdd[5]") 
Atdd = sp.Matrix([
    [Atxx,Atxy,Atxz],
    [Atxy,Atyy,Atyz],
    [Atxz,Atyz,Atzz]
])
betaX, betaY, betaZ = symbols("betau[0] betau[1] betau[2]") 
betau = sp.Matrix([betaX,betaY,betaZ])

BdX, BdY, BdZ = symbols("Bdriver[0] Bdriver[1] Bdriver[2]")
Bdriver = sp.Matrix([BdX,BdY,BdZ])

GammatX, GammatY, GammatZ = symbols("Gammatu[0] Gammatu[1] Gammatu[2]") 
Gammatu = sp.Matrix([GammatX, GammatY, GammatZ])

alpha, W = symbols("alp W")
theta, Khat = symbols("theta Khat", real=True)

# Hydro 
Strace, rho = symbols("S rho", real=True)
Sxx, Sxy, Sxz, Syy, Syz, Szz = symbols("Sij[0] Sij[1] Sij[2] Sij[3] Sij[4] Sij[5]", real=True)
Sdd = sp.Matrix(
    [
        [Sxx,Sxy,Sxz],
        [Sxy,Syy,Syz],
        [Sxz,Syz,Szz]
    ]
)
Sx, Sy, Sz = symbols("Si[0] Si[1] Si[2]", real=True)
Sd = sp.Matrix([Sx,Sy,Sz])

# Definition of K with Khat and theta 
Ktr_expl = Khat + S(2) * theta 
Ktr = symbols("Ktr", real=True)

# Parameters 
kappa1, kappa2 = symbols("kappa1 kappa2", real=True)
muL,muS,eta = symbols("muL muS eta", real=True)

# Aij up up 
Atuu_expl = sp.zeros(3,3)
Atuu = sp.zeros(3,3)
cidx = 0 
for a in range(3):
    for b in range(a,3):
        for c in range(3):
            for d in range(3):
                Atuu_expl[a,b] += sp.simplify(gtuu[a,c] * gtuu[b,d] * Atdd[c,d])
        Atuu[a,b] = Atuu[b,a] = symbols(f"Atuu[{cidx}]")
        Atuu_expl[b,a] = Atuu_expl[a,b]
        cidx += 1

AA = symbols("Asqr",real=True,nonnegative=True)
AA_expl = 0 
for a in range(3):
    for b in range(3):
        AA_expl += Atuu[a,b] * Atdd[a,b]

# Non conformal metric
gdd = gtdd/W**2

Matrix([[1, 0, 0], [0, 1, 0], [0, 0, 1]])


In [88]:
#================================================
# Derivatives
#================================================
dgtdd_dx = der_symm_tens(gtdd,"gtdd")
d2gtdd_dx2 = second_der_symm_tens(gtdd,"gtdd")
dgtdd_dx_upw = upwind_der_symm_tens(gtdd,"gtdd")

#================================================
dAtdd_dx_upw = upwind_der_symm_tens(Atdd,"Atdd")
dAtdd_dx = der_symm_tens(Atdd,"Atdd")

#================================================
dbetau_dx = der_vec(betau,"betau")
d2betau_dx2 = second_der_vec(betau,"betau")
dbetau_dx_upw = upwind_der_vec(betau,"betau")

#================================================
dBdriver_dx_upw = upwind_der_vec(Bdriver,"Bdriver")

#================================================
dGammatu_dx = der_vec(Gammatu,"Gammatu")
dGammatu_dx_upw = upwind_der_vec(Gammatu,"Gammatu")

#================================================
dalpha_dx = der_scalar(alpha,"alp")
d2alpha_dx2 = second_der_scalar(alpha,"alp")
dalpha_dx_upw = upwind_der_scalar(alpha,"alp")

#================================================
dKhat_dx = der_scalar(Khat,"Khat")
dKhat_dx_upw = upwind_der_scalar(Khat,"Khat")

#================================================
dW_dx = der_scalar(W, "W")
d2W_dx2 = second_der_scalar(W,"W")
dW_dx_upw = upwind_der_scalar(W,"W")

#================================================
dtheta_dx = der_scalar(theta, "theta")
dtheta_dx_upw = upwind_der_scalar(theta, "theta")

### Temporaries

Christoffel symbol (first kind):
$$
\tilde{\Gamma}_{ijk} = \frac{1}{2} \left( \partial_j \tilde{\gamma}_{ki} + \partial_k \tilde{\gamma}_{ji} - \partial_i \tilde{\gamma}_{jk} \right)
$$

In [89]:
Gammatddd_exp = sp.MutableDenseNDimArray.zeros(3,3,3)
Gammatddd = sp.MutableDenseNDimArray.zeros(3,3,3)
Gammatddd_flat = sp.zeros(18,1)

cidx = 0
for c in range(3):
    for a in range(3):
        for b in range(a,3):
            Gammatddd_exp[c,a,b] = Gammatddd_exp[c,b,a] = Rational(1,2) * (dgtdd_dx[a][b,c] + dgtdd_dx[b][a,c] - dgtdd_dx[c][a,b])
            Gammatddd[c,a,b] = Gammatddd[c,b,a] = symbols(f"Gammatddd[{cidx}]")
            Gammatddd_flat[cidx] = Gammatddd_exp[c,a,b]
            cidx +=1 

Raise one index:
$$
{\tilde{\Gamma}^i}_{jk} = \tilde{\gamma}^{il} \, \tilde{\Gamma}_{ljk}
$$

In [90]:
Gammatudd_expl = sp.MutableDenseNDimArray.zeros(3,3,3)
Gammatudd = sp.MutableDenseNDimArray.zeros(3,3,3)
Gammatudd_flat = sp.zeros(18,1)

cidx = 0
for c in range(3):
    for a in range(3):
        for b in range(a,3):
            for d in range(3):
                Gammatudd_expl[c,a,b] += gtuu[c,d] * Gammatddd[d,a,b]
            # Symmetrize
            Gammatudd_expl[c,b,a] = Gammatudd_expl[c,a,b]    
            # Symbolic 
            Gammatudd[c,b,a] = Gammatudd[c,a,b] = symbols(f"Gammatudd[{cidx}]")
            # Flatten for printing 
            Gammatudd_flat[cidx] = Gammatudd_expl[c,b,a]
            cidx += 1

Contracted Christoffel:
$$
(\tilde{\Gamma}_d)^k := \tilde{\gamma}^{ij} \, \tilde{\Gamma}^k_{ij}
$$

In [91]:
GammaDu = sp.zeros(3,1)
GammaDu_expl = sp.zeros(3,1)

for a in range(3):
    # Contract
    for b in range(3):
        for c in range(3):
            GammaDu_expl[a] += gtuu[b,c] * Gammatudd[a,b,c]
    # Symbolic 
    GammaDu[a] = symbols(f"GammatDu[{a}]")

For convenience we define here some derivatives of $\phi := \log \Psi$

$$
\partial_i \phi = -\frac{1}{2 W} \partial_i W
$$
And 
$$
\tilde{D}_i \tilde{D}_j \phi = \partial_i \partial_j \phi - {\tilde{\Gamma}^k}_{ij} \partial_k \phi = -\frac{1}{2\, W} \partial_i \partial_j W + 2 \, \partial_i \phi \partial_j \phi - {\tilde{\Gamma}^k}_{ij} \partial_k \phi
$$
$$
\tilde{D}_i \tilde{D}_j \phi = -\frac{1}{2 W} \partial_i \partial_j W + \frac{1}{2\, W^2} \partial_i W \partial_j W  + \frac{1}{2\, W} {\tilde{\Gamma}^k}_{ij} \, \partial_k W
$$

In [92]:
# --------------------------------------------------- #
dphi_dx = sp.zeros(3,1)
for a in range(3):
    dphi_dx[a] = - Rational(1,2) * dW_dx[a] / W 
# --------------------------------------------------- #
# --------------------------------------------------- #
DtiDtjphi = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        # --------------------------------------------------- #
        DtiDtjphi[a,b] = - Rational(1,2) * d2W_dx2[a][b]/(W) + S(2) * dphi_dx[a] * dphi_dx[b]
        # --------------------------------------------------- #
        for c in range(3):
            DtiDtjphi[a,b] -= Gammatudd[c,a,b] * dphi_dx[c]
        # --------------------------------------------------- #
        DtiDtjphi[b,a] = DtiDtjphi[a,b]
        # --------------------------------------------------- #

As a temporary we will store the double covariant derivative of the lapse and its trace.

The derivative reads:
$$
D_i D_j \alpha = \tilde{D}_i \tilde{D}_j \alpha - 2 \left( 2 \tilde{D}_{(i} \phi \, \tilde{D}_{j)} \alpha - \tilde{D}^k  \phi \tilde{D}_k \alpha \, \tilde{\gamma}_{ij} \right)
$$
Using the definition $W = \Psi^{-2}$ this becomes 
$$
D_i D_j \alpha = \tilde{D}_i \tilde{D}_j \alpha - 2 \left( - \tilde{D}_{(i} \log{W} \, \tilde{D}_{j)} \alpha +\frac{1}{2} \tilde{D}^k \log{W} \tilde{D}_k \alpha \tilde{\gamma}_{ij} \right)
$$
$$
D_i D_j \alpha = \tilde{D}_i \tilde{D}_j \alpha + \left( 2 \tilde{D}_{(i} \log{W} \, \tilde{D}_{j)} \alpha - \tilde{D}^k \log{W} \tilde{D}_k \alpha \tilde{\gamma}_{ij} \right)
$$

In [93]:
W2DiDjalp = sp.zeros(3,3)
cidx = 0 
for a in range(3):
    for b in range(a,3):   
        W2DiDjalp[a,b] = W2DiDjalp[b,a] = symbols(f"W2DiDjalp[{cidx}]")
        cidx+=1
        
# This is traced with the physical metric 
DiDialp_expl = 0 
for a in range(3):
    for b in range(3):
        # No W^2 because it's already included 
        DiDialp_expl += gtuu[a,b] * W2DiDjalp[a,b]

DiDialp = symbols("DiDialp")

In [94]:
# --------------------------------------------------- #
W2DiDjalp_expl = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):   
        # --------------------------------------------------- #
        # Dti Dtj alp = d_i d_j alp - Gammat^k_ij d_k alp 
        # --------------------------------------------------- #
        W2DiDjalp_expl[a,b] += d2alpha_dx2[a][b]
        # --------------------------------------------------- #
        for c in range(3):
            W2DiDjalp_expl[a,b] -= Gammatudd[c,a,b] * dalpha_dx[c]
        # --------------------------------------------------- #
        # Conformal factor terms 
        # --------------------------------------------------- #
        W2DiDjalp_expl[a,b] -= 2 * ( dphi_dx[a] * dalpha_dx[b] + dphi_dx[b] * dalpha_dx[a])
        # --------------------------------------------------- #
        for c in range(3):
            for d in range(3):
                W2DiDjalp_expl[a,b] += 2 * gtuu[c,d] * dphi_dx[c] * dalpha_dx[d] * gtdd[a,b]
        # --------------------------------------------------- #
        W2DiDjalp_expl[b,a] = W2DiDjalp_expl[a,b]
# --------------------------------------------------- #
# This is only used as W^2 D_i D_j alp so we avoid dividing and multiplying again and we just simplify here
# --------------------------------------------------- #
for a in range(3):
    for b in range(3):
        W2DiDjalp_expl[a,b] = sp.simplify(W**2*W2DiDjalp_expl[a,b])

We also store the Ricci tensor 
$$
R_{ij} = \tilde{R}_{ij} + R^{\chi}_{ij}
$$
First:
$$
\tilde{R}_{ij} = -\frac{1}{2} \tilde{\gamma}^{lm} \partial_l \partial_m \tilde{\gamma}_{ij} + \tilde{\gamma}_{k (i} \partial_{j)} \tilde{\Gamma}^k + (\tilde{\Gamma}_d)^k \tilde{\Gamma}_{(ij)k} + \tilde{\gamma}^{lm} \left( 2 {\tilde{\Gamma}^k}_{l(i} \tilde{\Gamma}_{j) km}  + {\tilde{\Gamma}^k}_{im} \tilde{\Gamma}_{klj} \right)
$$

In [95]:
# --------------------------------------------------- #
Rtdd_expl = sp.zeros(3,3)
# --------------------------------------------------- #
for a in range(3):
    for b in range(a,3):
        # --------------------------------------------------- #
        # Wave part 
        # --------------------------------------------------- #
        for c in range(3):
            for d in range(3):
                Rtdd_expl[a,b] -= Rational(1,2) * gtuu[c,d] * d2gtdd_dx2[c][d][a,b] 
        # --------------------------------------------------- #
        # Derivatives of Gammatilde
        # --------------------------------------------------- #
        for c in range(3):
            Rtdd_expl[a,b] += Rational(1,2) * (gtdd[c,a] * dGammatu_dx[b][c] 
                                             + gtdd[c,b] * dGammatu_dx[a][c])  
            Rtdd_expl[a,b] += Rational(1,2) * GammaDu[c] * (Gammatddd[a,b,c] + Gammatddd[b,a,c])
        # --------------------------------------------------- #
        # Contracted Christoffels 
        # --------------------------------------------------- #
        for c in range(3):
            for d in range(3):
                for e in range(3):
                    Rtdd_expl[a,b] += gtuu[c,d] * (Gammatudd[e,c,a] * Gammatddd[b,e,d] 
                                                 + Gammatudd[e,c,b] * Gammatddd[a,e,d] 
                                                 + Gammatudd[e,a,d] * Gammatddd[e,c,b])
        # --------------------------------------------------- #
        # Symmetrize 
        # --------------------------------------------------- #         
        Rtdd_expl[b,a] = Rtdd_expl[a,b]
# --------------------------------------------------- #
Rtdd_expl.is_symmetric() 
# --------------------------------------------------- #

True

Conformal part:
$$
R^\phi_{ij} = 4\,\tilde{D}_i \phi \tilde{D}_j \phi - 2 \tilde{D}_i \tilde{D}_j \phi - 2\, \left( \tilde{D}^k\tilde{D}_k \phi + 2 \tilde{D}_k \phi \tilde{D}^k \phi \right) \tilde{\gamma}_{ij}
$$

In [96]:
# --------------------------------------------------- #
W2Rphidd = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        W2Rphidd[a,b] = 4 * dphi_dx[a] * dphi_dx[b] - 2 * DtiDtjphi[a,b]
        for c in range(3):
            for d in range(3):
                W2Rphidd[a,b] -= 2 * gtuu[c,d] * (DtiDtjphi[c,d] + 2 * dphi_dx[c] * dphi_dx[d]) * gtdd[a,b]
        W2Rphidd[b,a] = W2Rphidd[a,b]
# --------------------------------------------------- #
# Ditto the comment for W^2 D_i D_j alp
# --------------------------------------------------- #
for a in range(3):
    for b in range(3):
        W2Rphidd[a,b] = sp.simplify(W2Rphidd[a,b] * W**2)
# --------------------------------------------------- #

Shibata NR book (2.230):
$$
W^2 R^W_{ij} = W \tilde{D}_i \tilde{D}_j W  + \tilde{\gamma}_{ij} \left( W\, \tilde{D}_k \tilde{D}^k W -2 \, \tilde{D}_k W \tilde{D}^k W \right)

In [97]:
# --------------------------------------------------- #
DtiDtjW = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        DtiDtjW[a,b] = d2W_dx2[a][b]
        for c in range(3):
            DtiDtjW[a,b] -= Gammatudd[c,a,b] * dW_dx[c]

        DtiDtjW[b,a] = DtiDtjW[a,b]
# --------------------------------------------------- #
W2RWdd = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        W2RWdd[a,b] += W*DtiDtjW[a,b]
        for c in range(3):
            for d in range(3):
                W2RWdd[a,b] += gtdd[a,b] * (W * DtiDtjW[c,d] - S(2) * dW_dx[c] * dW_dx[d]) * gtuu[c,d]
        W2RWdd[b,a] = W2RWdd[a,b]
# --------------------------------------------------- #

Check the two expressions match! 

In [98]:
print(sp.simplify(W2Rphidd-W2RWdd))

Matrix([[0, 0, 0], [0, 0, 0], [0, 0, 0]])


In [99]:
W2Rdd_expl = W**2*Rtdd_expl + W2Rphidd 

W2Rdd = sp.zeros(3,3)
cidx = 0 
for a in range(3):
    for b in range(a,3):
        W2Rdd[a,b] = W2Rdd[b,a] = symbols(f"W2Rdd[{cidx}]")
        cidx += 1

# Trace 
Rtrace_expl = 0 
Rtrace = symbols("Rtrace")
for a in range(3):
    for b in range(3):
        Rtrace_expl += W2Rdd[a,b] * gtuu[a,b]

Definition of ${\tilde{A}^i}_j$ 

In [100]:
Atud = sp.zeros(3,3)
for a in range(3):
    for b in range(3):
        for c in range(3):
            Atud[a,b] += gtuu[a,c] * Atdd[c,b]

## Constraints
Hamiltonian
$$
\mathcal{H} = R - \tilde{A}^{ab} \tilde{A}_{ab} + \frac{2}{3} K^2 - 16\, \pi \rho 
$$
Momentum constraint
$$
M^i = \partial_j \tilde{A}^{ij} + \tilde{\Gamma}^i_{jk} \tilde{A}^{jk} - \frac{2}{3} \tilde{\gamma}^{ij} \partial_j \left( \hat{K} + 2\Theta \right) - 3 \tilde{A}^{ij} \partial_j \log{W} - 8 \pi \tilde{\gamma}^{ij} S_j 
$$
Where
$$
\partial_j \tilde{A}^{ij} = \partial_j \left( \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \tilde{A}_{lk} \right) = \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \partial_j \tilde{A}_{lk} - \tilde{A}^{i}_k (\tilde{\Gamma}_d)^k + \tilde{A}^{j}_l \partial_j \tilde{\gamma}^{li} = \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \partial_j \tilde{A}_{lk} - \tilde{A}^{i}_k (\tilde{\Gamma}_d)^k -  \tilde{A}^{jl} \tilde{\gamma}^{ik} \partial_j \tilde{\gamma}_{lk} 
$$

In [101]:
# NB: some papers disagree on the signs here ... 
# Re-derived from ADM this seems to be correct, Hilditch+ 2012 has a typo
H_c = Rtrace - AA + Rational(2,3) * ( Khat + S(2)*theta)**2 - S(16) * sp.pi * rho

In [102]:
dAtuu_dx = sp.Matrix([0]*3)
for i in range(3):
    for j in range(3):
        dAtuu_dx[i] -= Atud[i,j] * GammaDu[j]
        for k in range(3):
            for l in range(3):
                dAtuu_dx[i] += gtuu[i,l] * gtuu[j,k] * dAtdd_dx[j][l,k] - Atuu[j,l] * gtuu[i,k] * dgtdd_dx[j][l,k]

# Note the factor of 3 in front of the A^ij d_j log W, ETK disagrees but I think this is correct @TODO check? 
# Gourgoullhon 2007 has 6 At^ij Dt_j log Psi which is -3 At^ij Dt_j log W... 
M_c = dAtuu_dx
for i in range(3):
    for j in range(3):
        M_c[i] += - Rational(2,3) * gtuu[i,j] * (dKhat_dx[j] + S(2) * dtheta_dx[j]) - 3 * Atuu[i,j] * dW_dx[j] / W - S(8) * sp.pi * gtuu[i,j] * Sd[j]
        for k in range(3):
            M_c[i] += Gammatudd[i][j,k] * Atuu[j,k]

## Equations

Evolution of chi (Shibata NR 2.227):
$$
\partial_t W = \frac{1}{3} W \left[ \alpha (\hat{K} + 2\Theta) - \partial_i \beta^i \right] + \beta^i \partial_i W 
$$

In [103]:
# --------------------------------------------------- #
dW_dt = Rational(1,3) * W * alpha * (Khat+S(2)*theta)  
# --------------------------------------------------- #
for a in range(3):
    dW_dt -= Rational(1,3) * W * dbetau_dx[a][a]
# --------------------------------------------------- #
dW_dt += dW_dx_upw
# --------------------------------------------------- #

Evolution of $\tilde{\gamma}_{ij}$ (Shibata NR 2.225)
$$
\partial_t \tilde{\gamma}_{ij} = - 2 \alpha \tilde{A}_{ij} + \beta^k \partial_k \tilde{\gamma}_{ij} + 2 \tilde{\gamma}_{k(i}\partial_{j)}\beta^k - \frac{2}{3} \tilde{\gamma}_{ij} \partial_k \beta^k
$$

In [104]:
# --------------------------------------------------- #
# --------------------------------------------------- #
dgtdd_dt = - 2 * alpha * Atdd 
# --------------------------------------------------- #
for a in range(3):
    dgtdd_dt -= Rational(2,3) * gtdd * dbetau_dx[a][a]
# --------------------------------------------------- #
for a in range(3):
    for b in range(3):
        for c in range(3):
            dgtdd_dt[a,b] += gtdd[c,a] * dbetau_dx[b][c] + gtdd[c,b] * dbetau_dx[a][c]
# --------------------------------------------------- #
dgtdd_dt += dgtdd_dx_upw
# --------------------------------------------------- #
# --------------------------------------------------- #
print(dgtdd_dt.is_symmetric())

True


Evolution of Khat (Shibata NR 2.258)
$$
\partial_t \hat{K} = - D^i D_i \alpha + \alpha \left[ \tilde{A}_{ij} \tilde{A}^{ij} + \frac{1}{3} \left( \hat{K} + 2\Theta \right)^2 \right] + 4 \pi \alpha \left( S + \rho \right) + \alpha \, \kappa_1 \,(1-\kappa_2) \, \Theta + \beta^i \partial_i \hat{K}

In [105]:
# --------------------------------------------------- #
dKhat_dt = -DiDialp 
# --------------------------------------------------- #
dKhat_dt += alpha*(AA + Rational(1,3)*Ktr**2)
# --------------------------------------------------- #
dKhat_dt += S(4) * sp.pi * alpha * (Strace+rho) 
# --------------------------------------------------- #
dKhat_dt += alpha * kappa1*(S(1)-kappa2)*theta 
# --------------------------------------------------- #
dKhat_dt += dKhat_dx_upw
# --------------------------------------------------- #

Evolution of $\tilde{\Gamma}^i$ (Shibata NR Eq. 2.259, Hilditch 2012 Eq. 5)
$$
\partial_t \tilde{\Gamma}^i = -2 \tilde{A}^{ij}\partial_j \alpha + 2 \alpha \left[ \tilde{\Gamma}^i_{jk} \tilde{A}^{jk} -3 \tilde{A}^{ij} \partial_j \log{W} - \frac{1}{3} \tilde{\gamma}^{ij} \partial_j \left( 2 \hat{K} + \Theta \right) - 8\pi \tilde{\gamma}^{ij} S_j \right] \\  + \tilde{\gamma}^{jk} \partial_j \partial_k \beta^i + \frac{1}{3} \tilde{\gamma}^{ij} \partial_j \partial_k \beta^k + \beta^j \partial_j \tilde{\Gamma}^i - (\tilde{\Gamma}_d)^j \partial_j \beta^i + \frac{2}{3} (\tilde{\Gamma}_d)^i \partial_j \beta^j - 2 \alpha \, \kappa_1 \left[ \tilde{\Gamma}^i - (\tilde{\Gamma}_d)^i \right]
$$ 

In [106]:
# ---------------------------------------------------
dGammatu_dt = - 2 * alpha * kappa1 * (Gammatu - GammaDu)
# ---------------------------------------------------
for a in range(3):
    for b in range(3):
        # ------------------
        dGammatu_dt[a] -= S(2) * Atuu[a,b] * dalpha_dx[b]  
        # ------------------
        dGammatu_dt[a] -= S(6) * alpha * Atuu[a,b] * dW_dx[b]/W
        # ------------------
        dGammatu_dt[a] -= Rational(2,3) * alpha * gtuu[a,b] * ( S(2) * dKhat_dx[b] + dtheta_dx[b] )
        # ------------------
        dGammatu_dt[a] -= S(16) * alpha * sp.pi * gtuu[a,b] * Sd[b]  # double check conffact here!
        # ------------------
        dGammatu_dt[a] -= GammaDu[b] * dbetau_dx[b][a]
        # ------------------
        dGammatu_dt[a] += Rational(2,3) * GammaDu[a] * dbetau_dx[b][b]
        # ------------------
        for c in range(3):
            # ------------------
            dGammatu_dt[a] += S(2) * alpha * Gammatudd[a,b,c] * Atuu[b,c]
            # ------------------
            dGammatu_dt[a] += gtuu[b,c] * d2betau_dx2[b][c][a]
            # ------------------
            dGammatu_dt[a] += Rational(1,3) * gtuu[a,b] * d2betau_dx2[b][c][c]
            # ------------------
dGammatu_dt += dGammatu_dx_upw
# ---------------------------------------------------
dGammatu_dt_s = sp.zeros(3,1)
for a in range(3):
    dGammatu_dt_s[a] = symbols(f"dGammatu_dt[{a}]")
# ---------------------------------------------------

Evolution of $\Theta$: (Hilditch 2012 Eq. 6), note we add a damping factor following Kawaguchi+ 2015
$$
\partial_t \Theta = \frac{1}{2} \alpha \left[ R - \tilde{A}_{ij} \tilde{A}^{ij} + \frac{2}{3} \left( \hat{K} + 2 \Theta \right)^2 \right] - \alpha \left[ 8\pi \rho + \kappa_1 (2 + \kappa_2) \Theta \right] + \beta^i \partial_i \Theta = \alpha \left( \frac{1}{2} \mathcal{H} - \kappa_1 ( 2 + \kappa_2 ) \Theta \right) + \beta^i \partial_i \Theta 
$$

In [107]:
theta_damp_fact = symbols("theta_damp_fact")

dtheta_dt = alpha * ( Rational(1,2) * H_c - kappa1 * (S(2) + kappa2) * theta )
dtheta_dt = theta_damp_fact * dtheta_dt + dtheta_dx_upw

Evolution of $\tilde{A}_{ij}$ (Shibata NR Eq. 2.226)
$$ 
\partial_t \tilde{A}_{ij} =  \Phi_{ij} + \alpha \left[ K \tilde{A}_{ij} - 2 {\tilde{A}^k}_i \tilde{A}_{kj} \right] + \beta^k \partial_k \tilde{A}_{ij} + 2 \tilde{A}_{k(i}\partial_{j)}\beta^k -\frac{2}{3} \tilde{A}_{ij} \partial_k \beta^k
$$
where we defined 
$$
\Phi_{ij} := W^2 \left[ -D_i D_j \alpha + \alpha (R_{ij} - 8 \pi S_{ij} )\right]^{\rm TF}
$$

In [108]:
Phi = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        Phi[a,b] += - W2DiDjalp[a,b] + alpha * (W2Rdd[a,b] - S(8) * sp.pi * W**2 * Sdd[a,b])
        Phi[a,b] += Rational(1,3) * gtdd[a,b] * ( DiDialp - alpha * (Rtrace - S(8) * sp.pi * Strace) )
        Phi[b,a] = Phi[a,b]


dAtdd_dt = alpha * Ktr * Atdd 

for a in range(3):
    for b in range(a,3):
        for c in range(3):
            # --
            dAtdd_dt[a,b] -= S(2) * alpha * Atud[c,a] * Atdd[c,b] 
            # --
            dAtdd_dt[a,b] += Atdd[c,a] * dbetau_dx[b][c] + Atdd[c,b] * dbetau_dx[a][c]
            # -- 
            dAtdd_dt[a,b] -= Rational(2,3) * Atdd[a,b] * dbetau_dx[c][c]
        # Symmetrize
        dAtdd_dt[b,a] = dAtdd_dt[a,b]
# -- 
dAtdd_dt += Phi + dAtdd_dx_upw

### Gauge sector

Lapse
$$
\partial_t \alpha = -\alpha^2 \mu_{\rm L} \hat{K} + \beta^i \partial_i \alpha 
$$
We choose $1+\log$ slicing: 
$$
\partial_t \alpha = -2 \alpha \hat{K} + \beta^i \partial_i \alpha 
$$
For the shift we use the strongly hyperbolic Gamma driver 
$$
\partial_t \beta^i = \frac{3}{4} B^i + \beta^k \partial_k \beta^i
$$
$$
\partial_t B^i = \partial_t \tilde{\Gamma}^i + \beta^k ( \partial_k B^i - \partial_k \tilde{\Gamma}^i ) - \eta B^i 
$$

In [109]:
##
dalpha_dt = -S(2) * alpha * Khat + dalpha_dx_upw
##
dbeta_dt = Rational(3,4) * Bdriver + dbetau_dx_upw
##
dBd_dt = dGammatu_dt_s + dBdriver_dx_upw - dGammatu_dx_upw - eta * Bdriver 

Following Lousto+2010 we can adapt eta to the conformal factor. They suggest:
$$
\eta = \eta_0 \frac{\sqrt{\tilde{\gamma}^{ij} \partial_i W \partial_j W}}{(1-W^a)^b} \, .
$$

In [110]:
apar,bpar,epstiny = symbols("apar bpar epstiny", real=True, nonnegative=True)
dchi2 = 0 
for a in range(3):
    for b in range(3):
        dchi2 += gtuu[a,b] * dW_dx[a] * dW_dx[b] 
eta_adaptive = Rational(1,2) * eta  * sp.sqrt(dchi2) / ((1-W**apar)**bpar + epstiny)

Now we write the matter source terms 
$$
\rho_{\rm ADM} = n^\mu n^\nu T_{\mu \nu}
$$
$$
S_i = -{\gamma^\mu}_i n^\nu T_{\mu\nu} 
$$
$$
S_{ij} = {\gamma^\mu}_i  {\gamma^\nu}_j T_{\mu\nu}
$$

In [111]:
betad = gdd * betau 
g4dd = sp.Matrix([
    [-alpha**2 + betau.dot(betad), betad[0], betad[1], betad[2]],
    [betad[0], gdd[0,0], gdd[0,1], gdd[0,2]],
    [betad[1], gdd[1,0], gdd[1,1], gdd[1,2]],
    [betad[2], gdd[2,0], gdd[2,1], gdd[2,2]],
])
guu = W**2 * gtuu 
g4uu = sp.zeros(4,4)
g4uu[0,0] = -1/alpha**2
for mu in range(3):
    g4uu[0,mu+1] = g4uu[mu+1,0] = betau[mu]/alpha**2
    for nu in range(3):
        g4uu[mu+1,nu+1] = guu[mu,nu] - betau[mu] * betau[nu] / alpha**2 

n4u = sp.Matrix([1/alpha,-betau[0]/alpha,-betau[1]/alpha,-betau[2]/alpha])
n4d = sp.Matrix([-alpha,0,0,0])

II4 = sp.Matrix([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
])
gamma_proj = II4 + n4u * n4d.T 

ttest = sp.zeros(4,4)
for a in range(4):
    for b in range(4):
        ttest[a,b] = II4[a,b] + n4u[a] * n4d[b]

print(sp.simplify(gamma_proj-ttest))

# Construct Tmunu 
rho0, P, eps = symbols("rho0 press eps", real=True)
zX, zY, zZ  = symbols("zvec[0] zvec[1] zvec[2]", real=True)
BX, BY, BZ  = symbols("Bvec[0] Bvec[1] Bvec[2]", real=True)
zvecu = sp.Matrix([zX,zY,zZ])
Bvecu = sp.Matrix([BX,BY,BZ])
Bvecd = gdd * Bvecu 
zvecd = gdd * zvecu 
B2 = Bvecu.dot(Bvecd)
# Basic derived qties 
h = 1 + eps + P/rho0
Lorentz = sp.sqrt(1+zvecu.dot(zvecd))
velu = zvecu / Lorentz 
veld = gdd * velu 
v2 = velu.dot(veld)

u0 = Lorentz / alpha 
u3u =  (alpha*velu-betau) * u0 
u4u = sp.Matrix([u0,*u3u])
u4d = g4dd * u4u 
u3d = sp.Matrix([u4d[1],u4d[2],u4d[3]])
# comoving B 
Bdotv = Bvecu.dot(veld)
bt_expl = Bvecu.dot(veld) * u0 
bi_expl = (Bvecu + alpha * bt_expl*u3u)/Lorentz
b2_expl = (B2 + alpha**2 * bt_expl**2)/Lorentz**2
b4u_expl = sp.Matrix([bt_expl,*bi_expl])
b4d = g4dd * b4u_expl

# Construct Tdownmunu 
Tdd = sp.zeros(4,4)

for mu in range(4):
    for nu in range(mu,4):
        Tdd[mu,nu] = Tdd[nu,mu] = (rho0*h + b2_expl) * u4d[mu] * u4d[nu] + (P + b2_expl/2) * g4dd[mu,nu] - b4d[mu] * b4d[nu]


rho_ADM = 0 
Sd_ADM = sp.zeros(3,1)
Sdd_ADM = sp.zeros(3,3)

for a in range(4):
    for b in range(4):
        rho_ADM +=  n4u[a] * n4u[b] * Tdd[a,b]

for a in range(3):
    for b in range(4):
        for c in range(4):
            Sd_ADM[a] += -gamma_proj[b,a+1] * n4u[c] * Tdd[b,c]

for i in range(3):
    for j in range(3):
        for mu in range(4):
            for nu in range(4):
                Sdd_ADM[i,j] += gamma_proj[mu,i+1] * gamma_proj[nu,j+1] * Tdd[mu,nu]

print(Sdd_ADM.is_symmetric())


trace_Sdd = sp.trace(guu*Sdd_ADM) 

Matrix([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
True


In [112]:
sp.simplify(rho_ADM - ( Lorentz**2*(rho0*h+b2_expl) - (P + b2_expl/S(2)) - alpha**2 *bt_expl**2 ))

For initial data: 
$$
\tilde{\Gamma}^i = -\partial_j \tilde{\gamma}^{ij} 
$$
Now:
$$
\partial_j \tilde{\gamma}^{ij}  = \partial_j (\tilde{\gamma}^{il} \tilde{\gamma}^{jm} \tilde{\gamma}_{lm}) = \tilde{\gamma}^{il} \tilde{\gamma}_{lm} \partial_j \tilde{\gamma}^{jm} + \tilde{\gamma}^{jm} \tilde{\gamma}_{lm} \partial_j \tilde{\gamma}^{il} + \tilde{\gamma}^{il} \tilde{\gamma}^{jm} \partial_j \tilde{\gamma}_{lm} = 2 \partial_j \tilde{\gamma}^{ij} + \tilde{\gamma}^{il} \tilde{\gamma}^{jm} \partial_j \tilde{\gamma}_{lm}
$$
So ultimately 
$$
- \partial_j \tilde{\gamma}^{ij} = \tilde{\gamma}^{il} \tilde{\gamma}^{jm} \partial_j \tilde{\gamma}_{lm}
$$

In [113]:
Gammatu_init = sp.zeros(3,1)
for i in range(3):
    for j in range(3):
        for l in range(3):
            for m in range(3):
                Gammatu_init[i] += gtuu[i,l] * gtuu[j,m] * dgtdd_dx[j][l,m]

## $\Psi_4$ calculation

Following Bruegmann+ 2018 calibration of moving puncture simulations

In [114]:
Ktr = (Khat+S(2)*theta)
Kdd = (Atdd + Rational(1,3) * Ktr*gtdd ) / W**2 

dgdd_dx = [] 
for idir in range(3):
    _dgdd = sp.zeros(3,3)
    for i in range(3):
        for j in range(3):
            _dgdd[i,j] = (dgtdd_dx[idir][i,j] - S(2) * gtdd[i,j] * dW_dx[idir]/W)/W**2
    dgdd_dx.append(_dgdd)

dKdd_dx = [] 
for idir in range(3):
    _dKdd = sp.zeros(3,3)
    dKtr = dKhat_dx[idir] + S(2) * dtheta_dx[idir]
    for i in range(3):
        for j in range(3):
            _dKdd[i,j] -= 2 * (Rational(1,3) * Ktr * gtdd[i,j] + Atdd[i,j] ) * dW_dx[idir]/W**3
            _dKdd[i,j] += Rational(1,3) * Ktr * dgtdd_dx[idir][i,j] / W**2 
            _dKdd[i,j] += Rational(1,3) * dKtr * gtdd[i,j] / W**2 
            _dKdd[i,j] += dAtdd_dx[idir][i,j] / W**2 
    dKdd_dx.append(_dKdd)

d2gdd_dx2 = sp.MutableDenseNDimArray.zeros(3,3,3,3)
for idir in range(3):
    for jdir in range(3):
        for l in range(3):
            for m in range(3):
                d2gdd_dx2[idir,jdir,l,m] += d2gtdd_dx2[idir][jdir][l,m]/W**2 
                d2gdd_dx2[idir,jdir,l,m] -= 2 * d2W_dx2[idir][jdir] * gtdd[l,m] / W**3
                d2gdd_dx2[idir,jdir,l,m] -= 2 * (dgtdd_dx[idir][l,m]*dW_dx[jdir] + dgtdd_dx[jdir][l,m]*dW_dx[idir])/W**3
                d2gdd_dx2[idir,jdir,l,m] += S(6) * dW_dx[idir] * dW_dx[jdir] * gtdd[l,m] / W**4

Gammaddd = sp.MutableDenseNDimArray.zeros(3,3,3)
for c in range(3):
    for a in range(3):
        for b in range(a,3):
            Gammaddd[c,a,b] = Gammaddd[c,b,a] = Rational(1,2) * (dgdd_dx[a][b,c] + dgdd_dx[b][a,c] - dgdd_dx[c][a,b])

Gammaudd = sp.MutableDenseNDimArray.zeros(3,3,3)
for a in range(3):
    for b in range(3):
        for c in range(3):
            for d in range(3):
                Gammaudd[a,b,c] += guu[a,d] * Gammaddd[d,b,c]

Gammau = sp.zeros(3,1)
for a in range(3):
    for b in range(3):
        for c in range(3):
            Gammau[a] += guu[b,c] * Gammaudd[a,b,c]

Ricci tensor
$$
R_{ab} = g^{cd} {\Gamma^e}_{ac} {\Gamma}_{ebd} - g^{cd} {\Gamma^e}_{ab} {\Gamma}_{ecd} + \frac{1}{2} g^{cd} \left( \partial_a \partial_c g_{bd} + \partial_b \partial_c g_{ad} - \partial_c \partial_d g_{ab} - \partial_a \partial_b g_{cd} \right)
$$

In [115]:
R3dd = sp.zeros(3,3)
for a in range(3):
    for b in range(a,3):
        for c in range(3):
            for d in range(3):
                # Wave 
                R3dd[a,b] += Rational(1,2) * guu[c,d] * (
                      d2gdd_dx2[a,c,b,d] + d2gdd_dx2[b,c,a,d]
                    - d2gdd_dx2[c,d,a,b] - d2gdd_dx2[a,b,c,d]
                )
                # Christoffel 
                for e in range(3):
                    R3dd[a,b] += guu[c,d] * Gammaudd[e,a,c] * Gammaddd[e,b,d]
                    R3dd[a,b] -= guu[c,d] * Gammaudd[e,a,b] * Gammaddd[e,c,d]
        R3dd[b,a] = R3dd[a,b]
                
R3tr = 0 
for a in range(3):
    for b in range(3):
        R3tr += guu[a,b]*R3dd[a,b]


DiKdd = [] 
for a in range(3):
    _DKdd = sp.zeros(3,3)
    for b in range(3):
        for c in range(3):
            _DKdd[b,c] = dKdd_dx[a][b,c]
            for d in range(3):
                _DKdd[b,c] -= Gammaudd[d,a,b] * Kdd[d,c]
                _DKdd[b,c] -= Gammaudd[d,a,c] * Kdd[b,d]
    DiKdd.append(_DKdd)

Riemann tensors. For hypersurface:
$$
^{(3)}{R}_{abcd} = g_{ac} R_{bd} + g_{bd} R_{ac} - g_{ad} R_{bc} - g_{bc} R_{ad} + \frac{1}{2} R \, \left( g_{ad} g_{bc} - g_{ac} g_{bd} \right)
$$
And for spacetime, using the Gauss relation (2.92) Gorgoullhon
$$
^{(4)}{R}_{abcd} = ^{(3)}{R}_{abcd} + K_{ac} K_{bd} - K_{ad} K_{bc}
$$
Notice all these indices are spatial, so the tensors are projected across $n^\alpha$ 

In [116]:
R3dddd = sp.MutableDenseNDimArray.zeros(3,3,3,3)
R4dddd = sp.MutableDenseNDimArray.zeros(3,3,3,3)

for a in range(3):
    for b in range(3):
        for c in range(3):
            for d in range(3):
                R3dddd[a,b,c,d] += gdd[a,c] * R3dd[b,d]
                R3dddd[a,b,c,d] += gdd[b,d] * R3dd[a,c]
                R3dddd[a,b,c,d] -= gdd[a,d] * R3dd[b,c]
                R3dddd[a,b,c,d] -= gdd[b,c] * R3dd[a,d]
                R3dddd[a,b,c,d] -= Rational(1,2) * gdd[a,c] * gdd[b,d] * R3tr
                R3dddd[a,b,c,d] += Rational(1,2) * gdd[a,d] * gdd[b,c] * R3tr

                R4dddd[a,b,c,d] = R3dddd[a,b,c,d]
                R4dddd[a,b,c,d] += Kdd[a,c] * Kdd[b,d]
                R4dddd[a,b,c,d] -= Kdd[a,d] * Kdd[b,c]

Contractions of Riemann with normal. 

One contraction (Codazzi relation):
$$
R_{abc} := ^{(4)}{R}_{abcd} n^d = D_b K_{ac} - D_c K_{ab}
$$

And two:
$$
^{(4)}{R}_{abcd} n^c n^d = R_{ab} + K K_{ab} - g^{cd} K_{ac} K_{db}
$$

In [117]:
# Contracted once with n 
R4n = sp.MutableDenseNDimArray.zeros(3,3,3)
for a in range(3):
    for b in range(3):
        for c in range(3):
            R4n[a,b,c] = DiKdd[b][a,c] - DiKdd[c][a,b]

# Contracted twice 
R4nn = sp.zeros(3,3)
for a in range(3):
    for b in range(3):
        R4nn[a,b] = R3dd[a,b] + Ktr * Kdd[a,b]
        for c in range(3):
            for d in range(3):
                R4nn[a,b] -= guu[c,d] * Kdd[a,c] * Kdd[d,b]

Tetrad construction

In [118]:
from sympy import LeviCivita

xc,yc,zc = symbols("xyz[0] xyz[1] xyz[2]", real=True) 

sqrtgdet = sp.sqrt(sp.det(gdd))

uvec = sp.zeros(3,1)
vvec = sp.zeros(3,1)
wvec = sp.zeros(3,1)

uvec[0] = xc
uvec[1] = yc 
uvec[2] = zc 

wvec[0] = -yc
wvec[1] = xc 
wvec[2] = 0

# v^i = g^{ia} eps_{abc} w^b u^c 
for i in range(3):
    for a in range(3):
        for b in range(3):
            for c in range(3):
                vvec[i] += guu[i,a] * sqrtgdet * LeviCivita(a,b,c) * wvec[b] * uvec[c]

# 1) normalize w 
wvec_sqr = 0 
for a in range(3):
    for b in range(3):
        wvec_sqr += gdd[a,b] * wvec[a]*wvec[b]

wvec_n = sp.zeros(3,1)
for a in range(3):
    wvec_n[a] = wvec[a]/sp.sqrt(wvec_sqr)

# 2) orthogonalize u and w 
udotw = 0 
for a in range(3):
    for b in range(3):
        udotw += gdd[a,b] * uvec[a] * wvec_n[b]

uvec_orth = sp.zeros(3,1)
for a in range(3):
    uvec_orth[a] = uvec[a] - udotw * wvec_n[a]

# 3) normalize u
uvec_sqr = 0 
for a in range(3):
    for b in range(3):
        uvec_sqr += gdd[a,b] * uvec_orth[a] * uvec_orth[b]

uvec_n = sp.zeros(3,1)
for a in range(3):
    uvec_n[a] = uvec_orth[a] / sp.sqrt(uvec_sqr)

# 4) orthogonalize v wrt u and w 
vdotw = 0 
vdotu = 0 
for a in range(3):
    for b in range(3):
        vdotu += gdd[a,b] * uvec_n[a] * vvec[b]
        vdotw += gdd[a,b] * wvec_n[a] * vvec[b]

vvec_orth = sp.zeros(3,1)
for a in range(3):
    vvec_orth[a] = vvec[a] - vdotu * uvec_n[a] - vdotw * wvec_n[a]

# 5) normalize 
vvec_sqr = 0
for a in range(3):
    for b in range(3):
        vvec_sqr += gdd[a,b] * vvec_orth[a] * vvec_orth[b]

vvec_n = sp.zeros(3,1)
for a in range(3):
    vvec_n[a] = vvec_orth[a] / sp.sqrt(vvec_sqr)


In [119]:
psi4Re = 0 
psi4Im = 0 

for a in range(3):
    for b in range(3):
        psi4Re += Rational(1,4) * R4nn[a,b] * ( vvec_n[a] * vvec_n[b] - wvec_n[a] * wvec_n[b] )
        psi4Im -= Rational(1,4) * R4nn[a,b] * ( vvec_n[a] * wvec_n[b] + vvec_n[b] * wvec_n[a] )
        for c in range(3):
            psi4Re -= Rational(1,2) * R4n[a,c,b] * uvec_n[c] * ( vvec_n[a] * vvec_n[b] - wvec_n[a] * wvec_n[b] )
            psi4Im += Rational(1,2) * R4n[a,c,b] * uvec_n[c] * ( vvec_n[a] * wvec_n[b] + vvec_n[b] * wvec_n[a] )
            for d in range(3):
                psi4Re += Rational(1,4) * R4dddd[d,a,c,b] * uvec_n[d] * uvec_n[c] * ( vvec_n[a] * vvec_n[b] - wvec_n[a] * wvec_n[b] )
                psi4Im -= Rational(1,4) * R4dddd[d,a,c,b] * uvec_n[d] * uvec_n[c] * ( vvec_n[a] * wvec_n[b] + vvec_n[b] * wvec_n[a] )

psi4Re *= sp.sqrt(xc**2+yc**2+zc**2)
psi4Im *= sp.sqrt(xc**2+yc**2+zc**2)

## ADM quantities

$$
E = \frac{1}{16 \pi} \int_{S} \sqrt{\gamma} \gamma^{ab} \gamma_{cd} \left( \partial_b \gamma_{ac} - \partial_c \gamma _{ab} \right) dS_d 
$$

$$
P_a = \frac{1}{8 \pi} \int_{S} \sqrt{\gamma} \left( K^b_a - \delta^b_a K \right) dS_b 
$$

$$
J_a = \frac{1}{8 \pi} \epsilon_{ac}^d \int_{S} \sqrt{\gamma} x^c \left( K^b_d - \delta^b_d K \right) dS_b 
$$

In [120]:
E_ADM = 0 
for a in range(3):
    for b in range(3):
        for c in range(3): 
            for d in range(3):
                E_ADM += guu[a,b] * gdd[c,d] * ( dgdd_dx[b][a,c] - dgdd_dx[c][a,b] ) 
sqrtgdet = W**(-1/6)
E_ADM *= sqrtgdet

In [121]:
# Write some code. ABI 
tens = ("double", [6])
scalar=("double",None)
vec = ("double",[3])
ABI = {
    "gtdd": tens,
    "Atdd": tens,
    "betau": vec,
    "alp": scalar,
    "W": scalar,
    "Bdriver": vec,
    "Gammatu": vec,
    "theta": scalar,
    "Khat": scalar,
    "Ktr": scalar,
    "S": scalar,
    "rho": scalar,
    "Sij": tens,
    "Si": vec,
    "kappa1": scalar,
    "kappa2": scalar,
    "eta": scalar,
    "detg": scalar,
    "theta_damp_fact": scalar,
    # Temporaries 
    "gtuu": ("double", [6]),
    "Atuu": tens,
    "Asqr": scalar,
    "Gammatddd": ( "double", [3*6]),
    "Gammatudd": ( "double", [3*6]),
    "Gammatudd": ( "double", [3*6]),
    "GammatDu": ("double", [3]),
    "W2DiDjalp": ("double", [6]),
    "DiDialp": scalar,
    "W2Rchi": tens,
    "W2Rdd": tens,
    "Rtrace": scalar,
    "dGammatu_dt": vec,
    # First derivatives
    "dgtdd_dx": ("double", [6*3]),
    "dgtdd_dx_upwind": ("double", [6]),
    "dAtdd_dx": ("double", [6*3]),
    "dAtdd_dx_upwind": ("double", [6]),
    "dbetau_dx": ("double", [3*3]),
    "dbetau_dx_upwind": ("double", [3]),
    "dBdriver_dx_upwind": ("double", [3]),
    "dGammatu_dx": ("double", [3*3]),
    "dGammatu_dx_upwind": ("double", [3]),
    "dKhat_dx": vec,
    "dKhat_dx_upwind": scalar,
    "dW_dx": vec,
    "dW_dx_upwind": scalar,
    "dalp_dx": vec,
    "dalp_dx_upwind": scalar,
    "dtheta_dx": vec,
    "dtheta_dx_upwind": scalar,
    # Second derivatives
    "ddgtdd_dx2": ("double", [6*6]),
    "ddAtdd_dx2": ("double", [6*6]),
    "ddbetau_dx2": ("double", [3*6]),
    "ddGammatu_dx2": ("double", [3*6]),
    "ddalp_dx2": ("double", [6]),
    "ddW_dx2": ("double", [6]),
    "ddtheta_dx2": ("double", [6]),
    "ddKhat_dx2": ("double", [6]),
    # Matter 
    "zvec": vec,
    "Bvec": vec,
    "rho0": scalar,
    "press": scalar,
    "eps": scalar,
    # Coordinates
    "xyz": vec,
    # Gamma driver params
    "apar": scalar,
    "bpar": scalar,
    "epstiny": scalar
}

In [122]:
cse_ignore_list = tuple(
    Gammatddd[a, b, c]
    for a in range(3)
    for b in range(3)
    for c in range(3)
)

cse_ignore_list_ext = tuple(
    Gammatddd[a, b, c]
    for a in range(3)
    for b in range(3)
    for c in range(3)
) + tuple(gtuu[a,b] for a in range(3) for b in range(3))

In [123]:
flist = [] 

# Temporaries 
out_list = ["detg"]
out_abi = {"detg": scalar}
exprs = [detg_expl]
flist.append(make_function(exprs,printer,"z4c_get_det_conf_metric",ABI,out_list,out_abi))

out_list = ["gtuu"]
out_abi = {"gtuu": tens}
#exprs = [sp.Matrix([gtuu_expl[0,0],gtuu_expl[0,1],gtuu_expl[0,2],gtuu_expl[1,1],gtuu_expl[1,2],gtuu_expl[2,2]])]
exprs = [gtuu_expl]
flist.append(make_function(exprs,printer,"z4c_get_inverse_conf_metric",ABI,out_list,out_abi))

out_list = ["Atuu"]
out_abi = {"Atuu": tens}
exprs = [sp.Matrix([Atuu_expl[0,0],Atuu_expl[0,1],Atuu_expl[0,2],Atuu_expl[1,1],Atuu_expl[1,2],Atuu_expl[2,2]])]
#exprs = [Atuu_expl]
flist.append(make_function(exprs,printer,"z4c_get_Atuu",ABI,out_list,out_abi,cse_ignore=tuple(gtuu[a,b] for a in range(3) for b in range(3))))

out_list = ["Asqr"]
out_abi = {"Asqr": scalar}
exprs = [AA_expl] 
#exprs = [Atuu_expl]
flist.append(make_function(exprs,printer,"z4c_get_Asqr",ABI,out_list,out_abi,cse_ignore=tuple(gtuu[a,b] for a in range(3) for b in range(3))))

out_list = ["Gammatddd"]
out_abi = {"Gammatddd": ("double", [18])}
exprs = [Gammatddd_flat]
flist.append(make_function(exprs,printer,"z4c_get_first_Christoffel",ABI,out_list,out_abi))

out_list = ["Gammatudd"]
out_abi = {"Gammatudd": ("double", [18])}
exprs = [Gammatudd_flat]
flist.append(make_function(exprs,printer,"z4c_get_second_Christoffel",ABI,out_list,out_abi))

out_list = ["GammatDu"]
out_abi = {"GammatDu": ("double", [3])}
exprs = [GammaDu_expl]
flist.append(make_function(exprs,printer,"z4c_get_contracted_Christoffel",ABI,out_list,out_abi))

out_list = ["W2DiDjalp"]
out_abi = {"W2DiDjalp": ("double", [6])}
#exprs = [DiDjalp_expl2]
exprs = [sp.Matrix([W2DiDjalp_expl[0,0],W2DiDjalp_expl[0,1],W2DiDjalp_expl[0,2],W2DiDjalp_expl[1,1],W2DiDjalp_expl[1,2],W2DiDjalp_expl[2,2]])]

flist.append(make_function(exprs,printer,"z4c_get_DiDjalp",ABI,out_list,out_abi,cse_ignore=cse_ignore_list_ext))

out_list = ["DiDialp"]
out_abi = {"DiDialp": scalar}
exprs = [DiDialp_expl]
flist.append(make_function(exprs,printer,"z4c_get_DiDialp",ABI,out_list,out_abi,cse_ignore=cse_ignore_list_ext))

out_list = ["W2Rtdd"]
out_abi = {"W2Rtdd": ("double", [6])}
#exprs = [Rw_expl]
exprs = [sp.Matrix([W**2*Rtdd_expl[0,0],W**2*Rtdd_expl[0,1],W**2*Rtdd_expl[0,2],W**2*Rtdd_expl[1,1],W**2*Rtdd_expl[1,2],W**2*Rtdd_expl[2,2]])]
flist.append(make_function(exprs,printer,"z4c_get_Ricci",ABI,out_list,out_abi, add_to_output=True))

out_list = ["W2Rphi"]
out_abi = {"W2Rphi": ("double", [6])}
#exprs = [Rphidd]
exprs = [sp.Matrix([W2Rphidd[0,0],W2Rphidd[0,1],W2Rphidd[0,2],W2Rphidd[1,1],W2Rphidd[1,2],W2Rphidd[2,2]])]

flist.append(make_function(exprs,printer,"z4c_get_Ricci_conf",ABI,out_list,out_abi,cse_ignore=cse_ignore_list, add_to_output=True))

out_list = ["Rtrace"]
out_abi = {"Rtrace": scalar}
exprs = [Rtrace_expl]

flist.append(make_function(exprs,printer,"z4c_get_Ricci_trace",ABI,out_list,out_abi,cse_ignore=cse_ignore_list))



# RHS 
out_list = ["dW"]
out_abi = {"dW": scalar}
exprs = [dW_dt]
flist.append(make_function(exprs,printer,"z4c_get_chi_rhs",ABI,out_list,out_abi))

out_list = ["dgtdd_dt"]
out_abi = {"dgtdd_dt": tens}
#exprs = [dgtdd_dt]
exprs = [sp.Matrix([dgtdd_dt[0,0],dgtdd_dt[0,1],dgtdd_dt[0,2],dgtdd_dt[1,1],dgtdd_dt[1,2],dgtdd_dt[2,2]])]

flist.append(make_function(exprs,printer,"z4c_get_gtdd_rhs",ABI,out_list,out_abi))

out_list = ["dKhat_dt"]
out_abi = {"dKhat_dt": scalar}
exprs = [dKhat_dt]
flist.append(make_function(exprs,printer,"z4c_get_Khat_rhs",ABI,out_list,out_abi))

out_list = ["dGammatu_dt"]
out_abi = {"dGammatu_dt": vec}
exprs = [dGammatu_dt]
flist.append(make_function(exprs,printer,"z4c_get_Gammatilde_rhs",ABI,out_list,out_abi,cse_ignore=cse_ignore_list_ext))

out_list = ["dtheta_dt"]
out_abi = {"dtheta_dt": scalar}
exprs = [dtheta_dt]
flist.append(make_function(exprs,printer,"z4c_get_theta_rhs",ABI,out_list,out_abi))

out_list = ["dAtdd_dt"]
out_abi = {"dAtdd_dt": tens}
#exprs = [dAtdd_dt]
exprs = [sp.Matrix([dAtdd_dt[0,0],dAtdd_dt[0,1],dAtdd_dt[0,2],dAtdd_dt[1,1],dAtdd_dt[1,2],dAtdd_dt[2,2]])]

flist.append(make_function(exprs,printer,"z4c_get_Atdd_rhs",ABI,out_list,out_abi))

out_list = ["dalpha_dt"]
out_abi = {"dalpha_dt": scalar}
exprs = [dalpha_dt]
flist.append(make_function(exprs,printer,"z4c_get_alpha_rhs",ABI,out_list,out_abi))

out_list = ["dbeta_dt"]
out_abi = {"dbeta_dt": vec}
exprs = [dbeta_dt]
flist.append(make_function(exprs,printer,"z4c_get_beta_rhs",ABI,out_list,out_abi))

out_list = ["dBd_dt"]
out_abi = {"dBd_dt": vec}
exprs = [dBd_dt]
flist.append(make_function(exprs,printer,"z4c_get_Bdriver_rhs",ABI,out_list,out_abi))

# Constraints 
out_list = ["H","M"]
out_abi = {"H": scalar, "M": vec}
exprs=[H_c,M_c]
flist.append(make_function(exprs,printer,"z4c_get_constraints",ABI,out_list,out_abi,cse_ignore=cse_ignore_list))

# Matter 
out_list = ["rho_adm","S_adm_trace", "Sd_adm","Sdd_adm"]
out_abi = {"rho_adm": scalar, "S_adm_trace": scalar, "Sd_adm": vec, "Sdd_adm": tens}
exprs=[rho_ADM,trace_Sdd,Sd_ADM,Sdd_ADM]
flist.append(make_function(exprs,printer,"z4c_get_matter_sources",ABI,out_list,out_abi))

# Gamma tilde ID
out_list = ["Gammatu_id"]
out_abi = {"Gammatu_id": vec}
exprs=[Gammatu_init]
flist.append(make_function(exprs,printer,"z4c_get_gammatilde_initial",ABI,out_list,out_abi))

# Psi4
out_list = ["Psi4Re","Psi4Im"]
out_abi = {"Psi4Re": scalar, "Psi4Im": scalar}
exprs=[psi4Re,psi4Im]
flist.append(make_function(exprs,printer,"z4c_get_psi4",ABI,out_list,out_abi,cse_order='none',cse_optims=None))

# Adaptive eta
out_list = ["eta_adaptive"]
out_abi = {"eta_adaptive": scalar}
exprs=[eta_adaptive]
flist.append(make_function(exprs,printer,"z4c_get_adaptive_eta",ABI,out_list,out_abi))

printed_functions = '\n'+'\n'.join(flist)


In [124]:
file = '''
/****************************************************************************/
/*                       Z4C helpers, SymPy generated                       */
/****************************************************************************/
#ifndef GRACE_Z4C_SUBEXPR_HH
#define GRACE_Z4C_SUBEXPR_HH

#include <Kokkos_Core.hpp>
''' + printed_functions + '''
#endif 
'''

with open("z4c_subexpressions.hh","w") as ff:
    ff.write(file)